# RL for Query Rewriting and Schema Selection

**Goal.** Train one phase-conditioned Actor-Critic agent to first decide whether and how to rewrite a natural-language question and then select the compact schema required for Text-to-Cypher.

The notebook is self-contained: it defines the dataset split, retriever, Gymnasium environment, neural policy, behavioral-cloning warm-up, end-to-end Actor-Critic training, baselines, ablations, and evaluation. No local Python module is imported.

## Research question

> Can one RL policy jointly learn *whether/how to rewrite* a question and *which schema triples to select*, while balancing retrieval quality, schema coverage, and action cost?

The gold schema is used only to create training demonstrations and rewards. It is never included in the agent observation at evaluation time.

In [13]:
from collections import Counter, deque
from copy import deepcopy
from pathlib import Path
import json, random, re

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAST_MODE = False         # Set True only for a short smoke test
BC_EPOCHS = 12 if FAST_MODE else 40
RL_EPISODES = 2000 if FAST_MODE else 10000
EVAL_EVERY = 200
MAX_REWRITE_TOKENS = 4
MAX_SCHEMA_ACTIONS = 5

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)
print("Device:", DEVICE)
print("Mode:", "FAST" if FAST_MODE else "FULL")

Device: cpu
Mode: FULL


## 1. Dataset and leakage-safe split

The split is stratified by hop count. Only the training split is indexed by the retriever. Validation selects checkpoints; test is evaluated once after all design choices are fixed.

In [14]:
DATA_PATH = Path("rl_schema_selection_project/data/schema_selection_dataset.json")
SUMMARY_PATH = Path("rl_schema_selection_project/data/schema_selection_summary.json")

with DATA_PATH.open(encoding="utf-8") as f:
    examples = json.load(f)
with SUMMARY_PATH.open(encoding="utf-8") as f:
    summary = json.load(f)

SCHEMA_TRIPLES = [
    (x["source"], x["relationship"], x["target"])
    for x in summary["schema_triples"]
]
N_SCHEMA = len(SCHEMA_TRIPLES)

def stratified_split(rows, seed=42, train_ratio=.70, val_ratio=.15):
    rng = random.Random(seed)
    groups = {}
    for row in rows:
        groups.setdefault(int(row["hop_count"]), []).append(row)
    train, val, test = [], [], []
    for group in groups.values():
        group = group.copy(); rng.shuffle(group)
        n_train = int(round(len(group) * train_ratio))
        n_val = int(round(len(group) * val_ratio))
        train.extend(group[:n_train])
        val.extend(group[n_train:n_train+n_val])
        test.extend(group[n_train+n_val:])
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

train_examples, val_examples, test_examples = stratified_split(examples, SEED)
print("Total/train/validation/test:", len(examples), len(train_examples), len(val_examples), len(test_examples))
print("Train hops:", Counter(x["hop_count"] for x in train_examples))
print("Validation hops:", Counter(x["hop_count"] for x in val_examples))
print("Test hops:", Counter(x["hop_count"] for x in test_examples))
print("Schema actions:", N_SCHEMA)

Total/train/validation/test: 484 339 73 72
Train hops: Counter({1: 179, 2: 147, 3: 13})
Validation hops: Counter({1: 38, 2: 32, 3: 3})
Test hops: Counter({1: 39, 2: 31, 3: 2})
Schema actions: 24


## 2. Text features, rewrite vocabulary, and retriever

The rewrite vocabulary contains all schema entity types and relations. A rewrite can either append selected schema tokens to the original question or replace the question with schema keywords while preserving quoted constants.

In [15]:
RELATION_CUES = {
    "TREATS_CtD": ["treat", "treated"], "PALLIATES_CpD": ["palliat", "alleviat"],
    "BINDS_CbG": ["bind", "binding"], "CAUSES_CcSE": ["cause", "side effect", "adverse"],
    "DOWNREGULATES_CdG": ["downregulat"], "UPREGULATES_CuG": ["upregulat"],
    "PRESENTS_DpS": ["symptom", "present"], "LOCALIZES_DlA": ["localiz", "anatomical location"],
    "ASSOCIATES_DaG": ["associat"], "RESEMBLES_DrD": ["similar", "resembl"],
    "RESEMBLES_CrC": ["similar", "resembl"], "INTERACTS_GiG": ["interact"],
    "COVARIES_GcG": ["co-vary", "covary", "covar"], "REGULATES_GrG": ["regulat"],
    "PARTICIPATES_GpPW": ["pathway", "participat"],
    "PARTICIPATES_GpBP": ["biological process", "participat"],
    "PARTICIPATES_GpMF": ["molecular function", "participat"],
    "PARTICIPATES_GpCC": ["cellular component", "participat"],
    "EXPRESSES_AeG": ["express"], "UPREGULATES_AuG": ["upregulat"],
    "DOWNREGULATES_AdG": ["downregulat"], "INCLUDES_PCiC": ["pharmacologic class", "include"],
}
ENTITY_CUES = {
    "Anatomy": ["anatom", "organ", "tissue"], "BiologicalProcess": ["biological process"],
    "CellularComponent": ["cellular component"], "Compound": ["compound", "drug"],
    "Disease": ["disease", "cancer", "condition"], "Gene": ["gene", "chromosome"],
    "MolecularFunction": ["molecular function"], "Pathway": ["pathway"],
    "PharmacologicClass": ["pharmacologic class"], "SideEffect": ["side effect", "adverse"],
    "Symptom": ["symptom"],
}

ENTITY_TOKENS = sorted({s for s, _, t in SCHEMA_TRIPLES} | {t for s, _, t in SCHEMA_TRIPLES})
RELATION_TOKENS = [r for _, r, _ in SCHEMA_TRIPLES]
REWRITE_TOKENS = ENTITY_TOKENS + RELATION_TOKENS
N_REWRITE = len(REWRITE_TOKENS)
TOKEN_TO_ID = {token: i for i, token in enumerate(REWRITE_TOKENS)}

def has_any(text, cues):
    text = text.lower()
    return float(any(cue in text for cue in cues))

FEATURE_NAMES = [
    "bias", "length", "quoted_count", "hop_2plus", "relation_cues", "entity_cues",
    "pathway", "biological_process", "molecular_function", "cellular_component",
    "compound", "disease", "gene", "anatomy", "bind", "similarity",
]

def question_features(question):
    q = question.lower()
    relation_matches = sum(has_any(q, cues) for cues in RELATION_CUES.values())
    entity_matches = sum(has_any(q, cues) for cues in ENTITY_CUES.values())
    quoted = re.findall(r"['\"]([^'\"]+)['\"]", question)
    conjunctions = sum(q.count(x) for x in [" and ", " or ", " that ", " which "])
    return np.asarray([
        1.0, min(len(q.split()) / 50, 1), min(len(quoted) / 3, 1), float(conjunctions >= 2),
        min(relation_matches / 4, 1), min(entity_matches / 4, 1),
        has_any(q, ["pathway"]), has_any(q, ["biological process"]),
        has_any(q, ["molecular function"]), has_any(q, ["cellular component"]),
        has_any(q, ["compound", "drug"]), has_any(q, ["disease", "cancer"]),
        has_any(q, ["gene", "chromosome"]), has_any(q, ["anatom", "organ", "tissue"]),
        has_any(q, ["bind"]), has_any(q, ["similar", "resembl"]),
    ], dtype=np.float32)

def quoted_terms(question):
    return re.findall(r"['\"]([^'\"]+)['\"]", question)

def build_rewrite(question, mode, selected_tokens):
    if mode == "keep" or not selected_tokens:
        return question
    token_text = " ".join(selected_tokens)
    if mode == "append":
        return f"{question} {token_text}".strip()
    constants = " ".join(quoted_terms(question))
    return f"{token_text} {constants}".strip()

def schema_text(example):
    return " ".join(" ".join(t) for t in example["gold_schema_triples"])

retrieval_documents = [f'{x["question"]} {schema_text(x)}' for x in train_examples]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
train_matrix = vectorizer.fit_transform(retrieval_documents)

def retrieve(query, k=3, excluded_train_index=None):
    scores = cosine_similarity(vectorizer.transform([query]), train_matrix).ravel()
    if excluded_train_index is not None:
        scores[excluded_train_index] = -np.inf
    ids = np.argsort(-scores)[:k]
    return [train_examples[i] for i in ids]

def retrieval_metrics(example, query, k=3, excluded_train_index=None):
    retrieved = retrieve(query, k, excluded_train_index)
    gold = set(map(tuple, example["gold_schema_triples"]))
    retrieved_schemas = [set(map(tuple, x["gold_schema_triples"])) for x in retrieved]
    exact = float(any(schema == gold for schema in retrieved_schemas))
    union = set().union(*retrieved_schemas)
    recall = len(gold & union) / max(1, len(gold))
    return exact, recall

def schema_compatibility_features(question):
    """One relevance feature per schema action, computed without gold labels."""
    q = question.lower()
    values = []
    for source, relation, target in SCHEMA_TRIPLES:
        relation_match = has_any(q, RELATION_CUES.get(relation, []))
        source_match = has_any(q, ENTITY_CUES.get(source, []))
        target_match = has_any(q, ENTITY_CUES.get(target, []))
        endpoint_pair = float(source_match and target_match)
        values.append(min(1.0, .55*relation_match + .15*source_match + .15*target_match + .15*endpoint_pair))
    return np.asarray(values, dtype=np.float32)

print("Rewrite tokens:", N_REWRITE)
print("Example retrieval:", retrieval_metrics(val_examples[0], val_examples[0]["question"]))

Rewrite tokens: 35
Example retrieval: (0.0, 0.5)


## 3. Unified Gymnasium MDP

One episode has two phases:

1. **REWRITE:** choose `append`/`keywords`, select schema tokens, then `STOP_REWRITE`.
2. **SCHEMA:** select schema triples, then `STOP_SCHEMA`.

The policy always emits logits over the same action space. An action mask exposes only phase-valid actions. `STOP_REWRITE` changes phase; only `STOP_SCHEMA` terminates the episode.

In [16]:
class UnifiedText2CypherEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, rows, split, seed=42, max_rewrite_tokens=4, max_schema_actions=5):
        super().__init__()
        self.rows, self.split = rows, split
        self.rng = random.Random(seed)
        self.max_rewrite_tokens = max_rewrite_tokens
        self.max_schema_actions = max_schema_actions

        self.SET_APPEND = N_REWRITE
        self.SET_KEYWORDS = N_REWRITE + 1
        self.STOP_REWRITE = N_REWRITE + 2
        self.SCHEMA_OFFSET = N_REWRITE + 3
        self.STOP_SCHEMA = self.SCHEMA_OFFSET + N_SCHEMA
        self.n_actions = self.STOP_SCHEMA + 1

        # q(original), q(current), current question-triple compatibility, masks, mode, phase, progress
        self.obs_dim = 2*len(FEATURE_NAMES) + N_SCHEMA + N_REWRITE + N_SCHEMA + 3 + 2 + 2
        self.action_space = spaces.Discrete(self.n_actions)
        self.observation_space = spaces.Box(0, 1, shape=(self.obs_dim,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None: self.rng.seed(seed)
        self.index = int(options["index"]) if options and "index" in options else self.rng.randrange(len(self.rows))
        self.example = self.rows[self.index]
        self.phase, self.mode = "rewrite", "keep"
        self.rewrite_mask = np.zeros(N_REWRITE, dtype=np.float32)
        self.schema_mask = np.zeros(N_SCHEMA, dtype=np.float32)
        self.rewrite_steps = self.schema_steps = 0
        return self._observation(), self._info()

    def selected_tokens(self):
        return [REWRITE_TOKENS[i] for i, x in enumerate(self.rewrite_mask) if x]

    def rewritten_query(self):
        return build_rewrite(self.example["question"], self.mode, self.selected_tokens())

    def schema_scores(self):
        gold = set(map(int, self.example["gold_schema_triple_ids"]))
        selected = {i for i, x in enumerate(self.schema_mask) if x}
        tp = len(gold & selected)
        precision = tp / len(selected) if selected else 0.0
        recall = tp / len(gold) if gold else 0.0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
        exact = float(selected == gold)
        return precision, recall, f1, exact

    def valid_action_mask(self):
        mask = np.zeros(self.n_actions, dtype=np.int8)
        if self.phase == "rewrite":
            if self.mode == "keep":
                mask[[self.SET_APPEND, self.SET_KEYWORDS, self.STOP_REWRITE]] = 1
            else:
                if self.rewrite_steps < self.max_rewrite_tokens:
                    mask[:N_REWRITE] = 1 - self.rewrite_mask.astype(np.int8)
                mask[self.STOP_REWRITE] = 1
        else:
            if self.schema_steps < self.max_schema_actions:
                mask[self.SCHEMA_OFFSET:self.SCHEMA_OFFSET+N_SCHEMA] = 1-self.schema_mask.astype(np.int8)
            mask[self.STOP_SCHEMA] = 1
        return mask

    def _observation(self):
        modes = np.asarray([self.mode == "keep", self.mode == "append", self.mode == "keywords"], np.float32)
        phases = np.asarray([self.phase == "rewrite", self.phase == "schema"], np.float32)
        progress = np.asarray([
            self.rewrite_steps/max(1, self.max_rewrite_tokens),
            self.schema_steps/max(1, self.max_schema_actions),
        ], np.float32)
        return np.concatenate([
            question_features(self.example["question"]), question_features(self.rewritten_query()),
            schema_compatibility_features(self.rewritten_query()),
            self.rewrite_mask, self.schema_mask, modes, phases, progress,
        ]).astype(np.float32)

    def _terminal_reward(self):
        precision, recall, f1, exact = self.schema_scores()
        # Retrieval receives its counterfactual reward at STOP_REWRITE.
        return 2.0*f1 + 1.0*exact - 0.10*(1-recall)

    def step(self, action):
        action = int(action)
        if not self.action_space.contains(action):
            raise ValueError(f"Action {action} is outside the discrete action space")
        if not self.valid_action_mask()[action]:
            # Masked actions are a penalized no-op. The trained policy masks
            # them before sampling, but handling them keeps the Gymnasium API total.
            return self._observation(), -1.0, False, False, self._info()
        reward, terminated, truncated = 0.0, False, False

        if self.phase == "rewrite":
            if action == self.SET_APPEND:
                self.mode = "append"; reward -= 0.005
            elif action == self.SET_KEYWORDS:
                self.mode = "keywords"; reward -= 0.005
            elif action == self.STOP_REWRITE:
                excluded = self.index if self.split == "train" else None
                original_exact, original_recall = retrieval_metrics(
                    self.example, self.example["question"], 3, excluded
                )
                rewrite_exact, rewrite_recall = retrieval_metrics(
                    self.example, self.rewritten_query(), 3, excluded
                )
                original_score = original_exact + .5*original_recall
                rewrite_score = rewrite_exact + .5*rewrite_recall
                rewrite_delta = rewrite_score - original_score
                reward += rewrite_delta
                if self.mode != "keep" and rewrite_delta <= 0:
                    reward -= 0.05
                self.phase = "schema"
            else:
                self.rewrite_mask[action] = 1
                self.rewrite_steps += 1
                reward -= 0.01
        else:
            if action == self.STOP_SCHEMA:
                terminated = True
                reward += self._terminal_reward()
            else:
                schema_id = action - self.SCHEMA_OFFSET
                old_f1 = self.schema_scores()[2]
                self.schema_mask[schema_id] = 1
                self.schema_steps += 1
                new_f1 = self.schema_scores()[2]
                # Dense incremental F1 feedback plus a small complexity cost.
                reward += (new_f1 - old_f1) - 0.025
        return self._observation(), float(reward), terminated, truncated, self._info()

    def _info(self):
        p, r, f1, exact = self.schema_scores()
        return {
            "phase": self.phase, "mode": self.mode, "selected_tokens": self.selected_tokens(),
            "rewritten_query": self.rewritten_query(), "selected_schema_ids": np.flatnonzero(self.schema_mask).tolist(),
            "schema_precision": p, "schema_recall": r, "schema_f1": f1, "schema_exact": exact,
            "schema_compatibility": schema_compatibility_features(self.rewritten_query()),
            "valid_action_mask": self.valid_action_mask(),
        }

    def action_name(self, action):
        if action < N_REWRITE: return REWRITE_TOKENS[action]
        if action == self.SET_APPEND: return "SET_APPEND"
        if action == self.SET_KEYWORDS: return "SET_KEYWORDS"
        if action == self.STOP_REWRITE: return "STOP_REWRITE"
        if action == self.STOP_SCHEMA: return "STOP_SCHEMA"
        return SCHEMA_TRIPLES[action-self.SCHEMA_OFFSET]

train_env = UnifiedText2CypherEnv(train_examples, "train", SEED)
obs, info = train_env.reset(options={"index": 0})
print("Gymnasium environment:", isinstance(train_env, gym.Env))
print("Observation/action spaces:", train_env.observation_space, train_env.action_space)
print("Initial valid actions:", [train_env.action_name(i) for i in np.flatnonzero(info["valid_action_mask"])])

Gymnasium environment: True
Observation/action spaces: Box(0.0, 1.0, (122,), float32) Discrete(63)
Initial valid actions: ['SET_APPEND', 'SET_KEYWORDS', 'STOP_REWRITE']


## 4. Environment sanity check

We execute one gold demonstration. Gold information is used here only to verify transitions and rewards; it is not part of the observation.

In [17]:
demo_env = UnifiedText2CypherEnv(train_examples, "train", SEED)
obs, info = demo_env.reset(options={"index": 0})
trajectory = [demo_env.STOP_REWRITE]
trajectory += [demo_env.SCHEMA_OFFSET+i for i in demo_env.example["gold_schema_triple_ids"]]
trajectory += [demo_env.STOP_SCHEMA]
total = 0
for action in trajectory:
    obs, reward, terminated, truncated, info = demo_env.step(action)
    total += reward
    print(demo_env.action_name(action), "reward=", round(reward, 3), "phase=", info["phase"])
print("Final F1/exact/total reward:", info["schema_f1"], info["schema_exact"], round(total, 3))

STOP_REWRITE reward= 0.0 phase= schema
('Compound', 'UPREGULATES_CuG', 'Gene') reward= 0.642 phase= schema
('Disease', 'ASSOCIATES_DaG', 'Gene') reward= 0.308 phase= schema
STOP_SCHEMA reward= 3.0 phase= schema
Final F1/exact/total reward: 1.0 1.0 3.95


## 5. One neural agent with phase-conditioned heads

The model has one shared encoder, one critic, and two actor heads. This remains one policy: the current MDP phase selects the relevant head, while the action mask enforces validity.

In [18]:
class UnifiedActorCritic(nn.Module):
    def __init__(self, obs_dim, hidden=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.rewrite_head = nn.Linear(hidden, N_REWRITE + 3)
        self.schema_head = nn.Linear(hidden, N_SCHEMA + 1)
        self.critic = nn.Linear(hidden, 1)

    def forward(self, obs):
        h = self.encoder(obs)
        rewrite_logits = self.rewrite_head(h)
        schema_logits = self.schema_head(h)
        logits = torch.cat([rewrite_logits, schema_logits], dim=-1)
        return logits, self.critic(h).squeeze(-1)

model = UnifiedActorCritic(train_env.obs_dim).to(DEVICE)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

UnifiedActorCritic(
  (encoder): Sequential(
    (0): Linear(in_features=122, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (2): Tanh()
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): Tanh()
  )
  (rewrite_head): Linear(in_features=128, out_features=38, bias=True)
  (schema_head): Linear(in_features=128, out_features=25, bias=True)
  (critic): Linear(in_features=128, out_features=1, bias=True)
)
Parameters: 40768


## 6. Behavioral cloning warm-up

For each training question, a pseudo-oracle compares `keep`, `append`, and `keywords` rewrites built from its gold schema. This is imitation learning on training demonstrations, not test-time leakage. The schema demonstration selects each gold triple and then stops.

In [19]:
def gold_rewrite_tokens(example):
    tokens = []
    for source, relation, target in map(tuple, example["gold_schema_triples"]):
        for token in (source, target, relation):
            if token in TOKEN_TO_ID and token not in tokens:
                tokens.append(token)
    return tokens[:MAX_REWRITE_TOKENS]

def pseudo_oracle_mode(example, train_index):
    tokens = gold_rewrite_tokens(example)
    candidates = [("keep", [])]
    if tokens:
        candidates += [("append", tokens), ("keywords", tokens)]
    scored = []
    for mode, selected in candidates:
        query = build_rewrite(example["question"], mode, selected)
        exact, recall = retrieval_metrics(example, query, 3, train_index)
        score = exact + .5*recall - .025*len(selected) - .01*(mode != "keep")
        scored.append((score, -len(selected), mode, selected))
    return max(scored, key=lambda x: (x[0], x[1]))[2:]

def demonstration(index):
    env = UnifiedText2CypherEnv(train_examples, "train", SEED)
    obs, info = env.reset(options={"index": index})
    mode, tokens = pseudo_oracle_mode(env.example, index)
    actions = []
    if mode == "append": actions.append(env.SET_APPEND)
    elif mode == "keywords": actions.append(env.SET_KEYWORDS)
    for token in tokens: actions.append(TOKEN_TO_ID[token])
    actions.append(env.STOP_REWRITE)
    actions += [env.SCHEMA_OFFSET+i for i in env.example["gold_schema_triple_ids"]]
    actions.append(env.STOP_SCHEMA)

    samples = []
    for action in actions:
        samples.append((obs.copy(), info["valid_action_mask"].copy(), action))
        obs, _, terminated, truncated, info = env.step(action)
    return samples

bc_samples = []
rewrite_demonstrations = 0
for i in range(len(train_examples)):
    demo = demonstration(i)
    rewrites = demo[0][2] != train_env.STOP_REWRITE
    rewrite_demonstrations += int(rewrites)
    # Rewriting is the minority decision. Oversampling its complete trajectory
    # prevents the rewrite head from learning the trivial always-keep policy.
    bc_samples.extend(demo * (3 if rewrites else 1))
print("Behavior-cloning state/action pairs:", len(bc_samples))
print("Training demonstrations that rewrite:", rewrite_demonstrations, "/", len(train_examples))
rewrite_bc_samples = [x for x in bc_samples if x[2] < train_env.SCHEMA_OFFSET]
print("Rewrite BC samples retained for RL regularization:", len(rewrite_bc_samples))

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
for epoch in range(1, BC_EPOCHS+1):
    random.shuffle(bc_samples)
    losses, correct = [], 0
    for start in range(0, len(bc_samples), 128):
        batch = bc_samples[start:start+128]
        obs = torch.tensor(np.stack([x[0] for x in batch]), dtype=torch.float32, device=DEVICE)
        masks = torch.tensor(np.stack([x[1] for x in batch]), dtype=torch.bool, device=DEVICE)
        targets = torch.tensor([x[2] for x in batch], dtype=torch.long, device=DEVICE)
        logits, _ = model(obs)
        masked_logits = logits.masked_fill(~masks, -1e9)
        loss = F.cross_entropy(masked_logits, targets)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item()); correct += (masked_logits.argmax(1) == targets).sum().item()
    print(f"BC epoch={epoch:02d} loss={np.mean(losses):.4f} accuracy={correct/len(bc_samples):.3f}")

bc_state = deepcopy(model.state_dict())

Behavior-cloning state/action pairs: 3098
Training demonstrations that rewrite: 92 / 339
Rewrite BC samples retained for RL regularization: 1765
BC epoch=01 loss=2.5463 accuracy=0.310
BC epoch=02 loss=2.2039 accuracy=0.406
BC epoch=03 loss=2.0062 accuracy=0.421
BC epoch=04 loss=1.8453 accuracy=0.454
BC epoch=05 loss=1.6854 accuracy=0.523
BC epoch=06 loss=1.5191 accuracy=0.590
BC epoch=07 loss=1.3722 accuracy=0.627
BC epoch=08 loss=1.2448 accuracy=0.659
BC epoch=09 loss=1.1320 accuracy=0.687
BC epoch=10 loss=1.0248 accuracy=0.709
BC epoch=11 loss=0.9509 accuracy=0.730
BC epoch=12 loss=0.8864 accuracy=0.751
BC epoch=13 loss=0.8133 accuracy=0.777
BC epoch=14 loss=0.7582 accuracy=0.789
BC epoch=15 loss=0.7068 accuracy=0.801
BC epoch=16 loss=0.6632 accuracy=0.813
BC epoch=17 loss=0.6200 accuracy=0.825
BC epoch=18 loss=0.5838 accuracy=0.835
BC epoch=19 loss=0.5577 accuracy=0.842
BC epoch=20 loss=0.5306 accuracy=0.857
BC epoch=21 loss=0.4905 accuracy=0.862
BC epoch=22 loss=0.4697 accuracy=0.8

## 7. Evaluation and baselines

The principal metrics are schema precision/recall/F1 and exact schema accuracy. We also report retrieval exact/recall, rewrite length, selected schema size, and total reward.

In [20]:
@torch.no_grad()
def greedy_action(model, obs, mask):
    x = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    logits, _ = model(x)
    mask = torch.tensor(mask, dtype=torch.bool, device=DEVICE)
    return int(logits.squeeze(0).masked_fill(~mask, -1e9).argmax().item())

def evaluate(model, rows, split, force_keep=False):
    env = UnifiedText2CypherEnv(rows, split, seed=999)
    records = []
    for i, example in enumerate(rows):
        obs, info = env.reset(options={"index": i}); total = 0
        for _ in range(MAX_REWRITE_TOKENS + MAX_SCHEMA_ACTIONS + 4):
            if force_keep and info["phase"] == "rewrite": action = env.STOP_REWRITE
            else: action = greedy_action(model, obs, info["valid_action_mask"])
            obs, reward, terminated, truncated, info = env.step(action); total += reward
            if terminated or truncated: break
        excluded = i if split == "train" else None
        rx, rr = retrieval_metrics(example, info["rewritten_query"], 3, excluded)
        records.append({
            "reward": total, "precision": info["schema_precision"], "recall": info["schema_recall"],
            "f1": info["schema_f1"], "schema_exact": info["schema_exact"], "retrieval_exact": rx,
            "retrieval_recall": rr, "rewrite_tokens": len(info["selected_tokens"]),
            "schema_size": len(info["selected_schema_ids"]), "mode": info["mode"],
        })
    numeric = [k for k in records[0] if k != "mode"]
    result = {k: float(np.mean([x[k] for x in records])) for k in numeric}
    result["mode_counts"] = dict(Counter(x["mode"] for x in records))
    return result, records

bc_validation, _ = evaluate(model, val_examples, "validation")
schema_only_validation, _ = evaluate(model, val_examples, "validation", force_keep=True)
print("BC unified validation:", bc_validation)
print("Schema-only ablation:", schema_only_validation)

BC unified validation: {'reward': 3.357739726027398, 'precision': 0.8835616438356164, 'recall': 0.8744292237442922, 'f1': 0.8707762557077625, 'schema_exact': 0.7808219178082192, 'retrieval_exact': 0.6438356164383562, 'retrieval_recall': 0.8744292237442922, 'rewrite_tokens': 1.452054794520548, 'schema_size': 1.4931506849315068, 'mode_counts': {'keep': 43, 'keywords': 15, 'append': 15}}
Schema-only ablation: {'reward': 3.390296803652968, 'precision': 0.9063926940639269, 'recall': 0.8858447488584473, 'f1': 0.8858447488584473, 'schema_exact': 0.7808219178082192, 'retrieval_exact': 0.589041095890411, 'retrieval_recall': 0.8904109589041096, 'rewrite_tokens': 0.0, 'schema_size': 1.4657534246575343, 'mode_counts': {'keep': 73}}


## 8. End-to-end Actor-Critic fine-tuning

We optimize discounted returns with a learned critic, entropy regularization, gradient clipping, validation checkpointing, and a decaying rewrite-imitation term. At `STOP_REWRITE`, a counterfactual reward compares rewritten and original retrieval quality. The terminal reward jointly evaluates retrieval and final schema; therefore early rewrite decisions receive delayed end-to-end credit.

In [21]:
def collect_episode(model, env):
    obs, info = env.reset()
    log_probs, values, rewards, entropies = [], [], [], []
    done = False
    while not done:
        x = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        logits, value = model(x); logits = logits.squeeze(0); value = value.squeeze(0)
        mask = torch.tensor(info["valid_action_mask"], dtype=torch.bool, device=DEVICE)
        dist = Categorical(logits=logits.masked_fill(~mask, -1e9))
        action = dist.sample()
        obs, reward, terminated, truncated, info = env.step(action.item())
        log_probs.append(dist.log_prob(action)); values.append(value)
        rewards.append(float(reward)); entropies.append(dist.entropy())
        done = terminated or truncated
    returns, running = [], 0.0
    for reward in reversed(rewards):
        running = reward + 0.99*running; returns.insert(0, running)
    returns = torch.tensor(returns, dtype=torch.float32, device=DEVICE)
    values = torch.stack(values); advantages = returns-values
    actor = -(torch.stack(log_probs)*advantages.detach()).mean()
    critic = F.mse_loss(values, returns)
    entropy = torch.stack(entropies).mean()
    loss = actor + .5*critic - .005*entropy
    return loss, sum(rewards), actor.item(), critic.item(), entropy.item()

model.load_state_dict(deepcopy(bc_state))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
best_state = deepcopy(model.state_dict())
best_validation = bc_validation
best_score = (bc_validation["f1"] + .25*bc_validation["retrieval_exact"]
              + .10*bc_validation["retrieval_recall"] - .01*bc_validation["rewrite_tokens"])
history, recent = [], deque(maxlen=200)

for episode in range(1, RL_EPISODES+1):
    loss, reward, actor_loss, critic_loss, entropy = collect_episode(model, train_env)

    # A decaying imitation term prevents catastrophic forgetting of useful
    # rewrite trajectories while leaving the RL objective dominant.
    beta = 0.10 - 0.09*(episode / RL_EPISODES)
    bc_batch = random.sample(rewrite_bc_samples, min(96, len(rewrite_bc_samples)))
    bc_obs = torch.tensor(np.stack([x[0] for x in bc_batch]), dtype=torch.float32, device=DEVICE)
    bc_masks = torch.tensor(np.stack([x[1] for x in bc_batch]), dtype=torch.bool, device=DEVICE)
    bc_targets = torch.tensor([x[2] for x in bc_batch], dtype=torch.long, device=DEVICE)
    bc_logits, _ = model(bc_obs)
    bc_loss = F.cross_entropy(bc_logits.masked_fill(~bc_masks, -1e9), bc_targets)
    total_loss = loss + beta*bc_loss

    optimizer.zero_grad(); total_loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
    recent.append(reward)

    if episode % EVAL_EVERY == 0:
        metrics, _ = evaluate(model, val_examples, "validation")
        score = (metrics["f1"] + .25*metrics["retrieval_exact"]
                 + .10*metrics["retrieval_recall"] - .01*metrics["rewrite_tokens"])
        if score > best_score:
            best_score, best_state, best_validation = score, deepcopy(model.state_dict()), metrics
        history.append({"episode": episode, "train_reward": np.mean(recent), **metrics})
        print(f"ep={episode:5d} trainR={np.mean(recent):.3f} valF1={metrics['f1']:.3f} "
              f"exact={metrics['schema_exact']:.3f} retExact={metrics['retrieval_exact']:.3f} "
              f"tokens={metrics['rewrite_tokens']:.2f} beta={beta:.3f} best={best_score:.3f}")

model.load_state_dict(best_state)
print("Best validation:", best_validation)

ep=  200 trainR=2.758 valF1=0.823 exact=0.616 retExact=0.603 tokens=0.08 beta=0.098 best=1.105
ep=  400 trainR=2.360 valF1=0.860 exact=0.699 retExact=0.589 tokens=0.21 beta=0.096 best=1.105
ep=  600 trainR=2.542 valF1=0.879 exact=0.726 retExact=0.575 tokens=0.25 beta=0.095 best=1.109
ep=  800 trainR=2.582 valF1=0.901 exact=0.781 retExact=0.589 tokens=0.21 beta=0.093 best=1.137
ep= 1000 trainR=2.519 valF1=0.895 exact=0.795 retExact=0.589 tokens=0.15 beta=0.091 best=1.137
ep= 1200 trainR=2.513 valF1=0.904 exact=0.781 retExact=0.575 tokens=0.19 beta=0.089 best=1.137
ep= 1400 trainR=2.790 valF1=0.874 exact=0.726 retExact=0.603 tokens=0.08 beta=0.087 best=1.137
ep= 1600 trainR=2.700 valF1=0.888 exact=0.767 retExact=0.603 tokens=0.08 beta=0.086 best=1.137
ep= 1800 trainR=2.599 valF1=0.888 exact=0.767 retExact=0.589 tokens=0.34 beta=0.084 best=1.137
ep= 2000 trainR=2.716 valF1=0.913 exact=0.808 retExact=0.589 tokens=0.38 beta=0.082 best=1.147
ep= 2200 trainR=2.751 valF1=0.890 exact=0.767 retE

## 9. Final ablations and locked test evaluation

We compare the jointly trained policy with (a) its behavioral-cloning initialization and (b) the same policy forced to skip rewriting. The test set should be evaluated only after fixing the complete protocol.

In [22]:
model.load_state_dict(bc_state)
bc_test, _ = evaluate(model, test_examples, "test")

model.load_state_dict(best_state)
unified_test, unified_records = evaluate(model, test_examples, "test")
schema_only_test, _ = evaluate(model, test_examples, "test", force_keep=True)

def row(name, x):
    return [name, x["f1"], x["schema_exact"], x["precision"], x["recall"],
            x["retrieval_exact"], x["retrieval_recall"], x["rewrite_tokens"], x["schema_size"]]

table = [
    row("Behavioral cloning", bc_test),
    row("Unified Actor-Critic", unified_test),
    row("Unified AC, no rewrite", schema_only_test),
]
header = ["Method", "Schema F1", "Schema exact", "P", "R", "Ret. exact", "Ret. recall", "Tokens", "Schema size"]
print(" | ".join(f"{h:>16}" for h in header))
print("-"*160)
for values in table:
    print(f"{values[0]:>24} | " + " | ".join(f"{v:16.3f}" for v in values[1:]))

          Method |        Schema F1 |     Schema exact |                P |                R |       Ret. exact |      Ret. recall |           Tokens |      Schema size
----------------------------------------------------------------------------------------------------------------------------------------------------------------
      Behavioral cloning |            0.865 |            0.694 |            0.912 |            0.845 |            0.583 |            0.850 |            1.042 |            1.347
    Unified Actor-Critic |            0.920 |            0.792 |            0.958 |            0.907 |            0.583 |            0.863 |            0.167 |            1.361
  Unified AC, no rewrite |            0.916 |            0.778 |            0.951 |            0.907 |            0.569 |            0.850 |            0.000 |            1.375


## 10. Results by hop and reproducible checkpoint

Performance by hop tests whether longer compositional questions remain harder. The checkpoint includes the model, split IDs, configuration, vocabulary, validation result, and test result.

In [23]:
for hop in sorted(set(x["hop_count"] for x in test_examples)):
    ids = [i for i, x in enumerate(test_examples) if x["hop_count"] == hop]
    subset = [unified_records[i] for i in ids]
    print({
        "hop": hop, "n": len(ids),
        "f1": round(float(np.mean([x["f1"] for x in subset])), 3),
        "exact": round(float(np.mean([x["schema_exact"] for x in subset])), 3),
        "rewrite_tokens": round(float(np.mean([x["rewrite_tokens"] for x in subset])), 3),
    })

artifact = {
    "model_state_dict": best_state,
    "schema_triples": SCHEMA_TRIPLES,
    "rewrite_tokens": REWRITE_TOKENS,
    "seed": SEED,
    "fast_mode": FAST_MODE,
    "validation": best_validation,
    "test": unified_test,
    "split_ids": {
        "train": [x["id"] for x in train_examples],
        "validation": [x["id"] for x in val_examples],
        "test": [x["id"] for x in test_examples],
    },
}
torch.save(artifact, "unified_rl_text2cypher_checkpoint.pt")
with open("unified_rl_text2cypher_results.json", "w", encoding="utf-8") as f:
    json.dump({"validation": best_validation, "test": unified_test, "history": history}, f, indent=2)
print("Saved checkpoint and results.")

{'hop': 1, 'n': 39, 'f1': 0.983, 'exact': 0.949, 'rewrite_tokens': 0.308}
{'hop': 2, 'n': 31, 'f1': 0.849, 'exact': 0.645, 'rewrite_tokens': 0.0}
{'hop': 3, 'n': 2, 'f1': 0.8, 'exact': 0.0, 'rewrite_tokens': 0.0}
Saved checkpoint and results.
